# 1- Initialize Spark Session with Hive support enabled


In [2]:
# Initialize Spark Session with Hive support enabled
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Streaming_temperature_lab2") \ # Num of Partitions in Spark by default is 200 so we choose 2 only becasue the data is small
    .config("spark.sql.shuffle.partitions", "2") \
    .config("spark.sql.warehouse.dir", "/home/itversity/itversity-material/learn_spark/spark-warehouse") \
    .enableHiveSupport() \
    .getOrCreate()

spark

# 2- Define schema matching the JSON files


In [3]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

schema = StructType([
    StructField("event_timestamp", StringType(), True),
    StructField("country", StringType(), True),
    StructField("temperature", DoubleType(), True)
])
print(schema)

StructType(List(StructField(event_timestamp,StringType,true),StructField(country,StringType,true),StructField(temperature,DoubleType,true)))


# 3- Read streaming files 

In [4]:
# We must use file:// prefix to specify local path inside the container and option("multiLine", "true")
input_path = "file:///home/itversity/itversity-material/learn_spark/lab2"

df = spark.readStream \
    .schema(schema) \
    .option("multiLine", "true") \
    .json(input_path)

# 4- Transform the data:
#### 1. Parse string timestamp to Timestamp type
#### 2. Add watermark of 10 minutes to handle late data
#### 3. Aggregate average temperature per country in 15-minute tumbling windows

In [5]:
from pyspark.sql.functions import col, to_timestamp, window, avg

df_transformed = df \
    .withColumn("timestamp", to_timestamp(col("event_timestamp"), "yyyy-MM-dd HH:mm:ss")) \
    .withWatermark("timestamp", "10 minutes") \
    .groupBy(
        window(col("timestamp"), "15 minutes"),
        col("country")
    ) \
    .agg(avg("temperature").alias("avg_temperature")) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("country"),
        col("avg_temperature")
    )

# 5- Load the data to: 
#### 1. Local filesystem in Parquet format
#### 2. Hive table (lab2_db.temperature_avg)


In [6]:
# Define the foreachBatch writer function to output data to 2 different locations:

output_fs = "file:///home/itversity/itversity-material/learn_spark/lab2_output/filesystem"
checkpoint_dir = "file:///home/itversity/itversity-material/learn_spark/lab2_output/checkpoint"

def write_to_sinks(batch_df, batch_id):
    print(f"\n=== [Batch {batch_id}] Writing Finalized Windowed Aggregations ===")
    batch_df.show(truncate=False)
    
    # Location 1: Write to Filesystem as Parquet
    batch_df.write \
        .mode("append") \
        .parquet(output_fs)
    
    # Location 2: Write to Hive Table
    spark.sql("CREATE DATABASE IF NOT EXISTS lab2_db")
    batch_df.write \
        .mode("append") \
        .saveAsTable("lab2_db.temperature_avg")

In [7]:
# Start the streaming query using Append output mode
# Append is the correct mode when performing windowed aggregations with watermark to be written to file sinks
query = df_transformed.writeStream \
    .foreachBatch(write_to_sinks) \
    .outputMode("append") \
    .option("checkpointLocation", checkpoint_dir) \
    .start()

In [8]:
# Wait for some time to let Spark process all existing files, then stop the query
import time
time.sleep(20)

query.stop()
query.awaitTermination()
print("Streaming query stopped successfully.")

Streaming query stopped successfully.


In [9]:
# Verify Filesystem Output (Parquet)
fs_df = spark.read.parquet(output_fs)
print(f"Total rows in Parquet destination: {fs_df.count()}")
fs_df.orderBy("window_start", "country").show(truncate=False)

Total rows in Parquet destination: 82
+-------------------+-------------------+---------+---------------+
|window_start       |window_end         |country  |avg_temperature|
+-------------------+-------------------+---------+---------------+
|2024-01-15 10:00:00|2024-01-15 10:15:00|Japan    |8.7            |
|2024-01-15 10:00:00|2024-01-15 10:15:00|UK       |5.2            |
|2024-01-15 10:00:00|2024-01-15 10:15:00|USA      |12.5           |
|2024-01-15 10:15:00|2024-01-15 10:30:00|Brazil   |28.9           |
|2024-01-15 10:15:00|2024-01-15 10:30:00|France   |6.8            |
|2024-01-15 10:15:00|2024-01-15 10:30:00|Germany  |3.4            |
|2024-01-15 10:15:00|2024-01-15 10:30:00|India    |22.5           |
|2024-01-15 10:15:00|2024-01-15 10:30:00|UK       |5.0            |
|2024-01-15 10:15:00|2024-01-15 10:30:00|USA      |13.1           |
|2024-01-15 10:30:00|2024-01-15 10:45:00|Australia|31.2           |
|2024-01-15 10:30:00|2024-01-15 10:45:00|Germany  |3.8            |
|2024-01-1

In [10]:
# Verify Hive Table Output
try:
    hive_df = spark.sql("SELECT * FROM lab2_db.temperature_avg ORDER BY window_start, country")
    print(f"Total rows in Hive table: {hive_df.count()}")
    hive_df.show(truncate=False)
except Exception as e:
    print(f"[ERROR] Failed to query Hive table: {e}")

Total rows in Hive table: 82
+-------------------+-------------------+---------+---------------+
|window_start       |window_end         |country  |avg_temperature|
+-------------------+-------------------+---------+---------------+
|2024-01-15 10:00:00|2024-01-15 10:15:00|Japan    |8.7            |
|2024-01-15 10:00:00|2024-01-15 10:15:00|UK       |5.2            |
|2024-01-15 10:00:00|2024-01-15 10:15:00|USA      |12.5           |
|2024-01-15 10:15:00|2024-01-15 10:30:00|Brazil   |28.9           |
|2024-01-15 10:15:00|2024-01-15 10:30:00|France   |6.8            |
|2024-01-15 10:15:00|2024-01-15 10:30:00|Germany  |3.4            |
|2024-01-15 10:15:00|2024-01-15 10:30:00|India    |22.5           |
|2024-01-15 10:15:00|2024-01-15 10:30:00|UK       |5.0            |
|2024-01-15 10:15:00|2024-01-15 10:30:00|USA      |13.1           |
|2024-01-15 10:30:00|2024-01-15 10:45:00|Australia|31.2           |
|2024-01-15 10:30:00|2024-01-15 10:45:00|Germany  |3.8            |
|2024-01-15 10:30:0